# The Precinct Goes Automated
Notebook 5: Workspace Monitoring Dashboard
> *"I know what happened here. I just need the dashboard to confirm it."* — Adrian Monk
This notebook demonstrates `create_workspace_monitoring_dashboard` from `sempy_labs.kql_dashboard`.
In **one function call**, Semantic Link Labs creates a pre-built monitoring dashboard scoped to your workspace.
> ⚠️ **Pre-requisite:** Workspace monitoring must be enabled before running this notebook.
> Enable it in: **Workspace Settings → Administration → Workspace monitoring.**
> Without it, the function raises: `"Workspace monitoring is not set up for this workspace."`
> `00_Setup_Fresh` installs the latest SLL version automatically — no separate upgrade step needed.

---
## Setup

In [ ]:
%run 00_Setup_Fresh

In [ ]:
%pip show semantic-link-labs

In [ ]:
from sempy_labs.kql_dashboard import create_workspace_monitoring_dashboard

---
## One Call. That's It.
 > *"The precinct doesn't run itself. But now it kind of does."*
 With workspace monitoring enabled, one call creates the dashboard — no configuration required.
 `workspace=None` (the default) resolves to the current workspace. No argument needed.

In [ ]:
# 👇 One call. That's it.
# workspace defaults to None — resolves to the current workspace automatically.
create_workspace_monitoring_dashboard()

---
## One-Time Configuration After First Run
 After the dashboard is created, open it in the workspace and click **Edit** to enter edit mode.
Under **Manage → Data sources**, select the **Monitoring KQL database** as the data source.
 Choose a data source access permission:
- **Pass-through identity** — each viewer queries using their own identity
- **Dashboard editor's identity** — dashboard queries run as the editor; viewers don't need direct KQL access
 Set it once. The dashboard is ready for every run after that.

---
## What Was Created
# After running the cell above, open your Fabric workspace and you'll find:
# | Item | Type | Description |
|------|------|-------------|
| Workspace Monitoring Dashboard | Real-Time Dashboard | Pre-built views wired to the existing Monitoring KQL database |
# The dashboard surfaces:
- **Capacity utilization** over time
- **Semantic model refresh** history and failures
- **Query activity** across datasets
- **Workspace item inventory**
# > The underlying KQL infrastructure (Eventhouse + Monitoring database) is created when workspace monitoring
> is enabled — not by this function. Once enabled, telemetry streams automatically.
No refresh schedule. No pipeline. No manual wiring required to the dashboard itself.

---
## The Visitor Log — Who Was in the Building?
 > *"Monk doesn't just look at the crime scene. He checks who had access. `list_activity_events` is your visitor log."*
 `labs.admin.list_activity_events()` pulls the **Fabric Activity Events** audit log for your workspace —
every refresh, report view, model change, and user action, with timestamps and identities.
What it reveals:
- Who refreshed a dataset and when
- Who viewed which report
- Who created, deleted, or modified a workspace item
- Patterns that correlate with "it was fine on Friday" complaints
 > ⚠️ **Requires Fabric Admin role** — Contributor access is not sufficient.
> If you receive a 403, ask your Fabric Admin to run this, or use a service principal scoped to Admin APIs.

In [ ]:
from sempy_labs import admin
from datetime import datetime, timezone, timedelta

# offset_days=0 → today (demo day), offset_days=1 → yesterday (for testing)
# API requires start and end to be within the same UTC day
offset_days = 0

target_date = (datetime.now(timezone.utc) - timedelta(days=offset_days)).date()
start_time = f"{target_date}T00:00:00"
end_time   = f"{target_date}T23:59:59"

print(f"Pulling activity for: {target_date}")

df_activity = admin.list_activity_events(
    start_time=start_time,
    end_time=end_time
)

print(f"Events returned: {len(df_activity)}")
display(df_activity)

### Filter for the operations that matter
 The activity log captures many event types. Common operations to filter on:
 | `Operation` value | What it means |
|---|---|
| `RefreshDataset` | Someone or a pipeline triggered a model refresh |
| `ViewReport` | A user opened a report |
| `CreateReport` / `DeleteReport` | Report created or deleted |
| `UpdateDataset` | Model was modified |
| `ExportReport` | Report exported (PDF, PowerPoint, etc.) |

In [ ]:
# Filter to model refreshes only — who ran them and when?
# Note: SLL renames API columns — "Creation Time" not "CreationTime", "User Id" not "UserId", etc.
if not df_activity.empty and "Operation" in df_activity.columns:
    df_refreshes = df_activity[df_activity["Operation"] == "RefreshDataset"]
    display(df_refreshes[["Creation Time", "User Id", "Artifact Name", "Operation"]].head(20))
else:
    print("No activity events returned — confirm Fabric Admin role and date range.")

---
### Dashboard + Log — The Complete Picture
 | Tool | What it shows |
|---|---|
| `create_workspace_monitoring_dashboard()` | Live telemetry — what is happening **right now** |
| `list_activity_events()` | Audit trail — what **already happened** and who did it |
 Together: you know the current state of the workspace AND its history.
That's the precinct. That's the full detective toolkit.
 > *"I know who was here. I know what they did. I know when."* — Monk (paraphrased)
 ---
 > *"Every clue, catalogued. Every anomaly, surfaced. Monk approves."*